# Análisis de Ventas y Rentabilidad — Business Sales Analytics

**Caso empresarial sintético · Chile · enero 2024 a diciembre 2025**

Los datos y resultados son simulados. Este informe muestra un flujo reproducible para evaluar ventas, utilidad bruta y decisiones comerciales.

## 1. Resumen ejecutivo

Se calculan KPIs desde transacciones limpias, se comparan productos, regiones, canales y descuentos, y se formulan acciones verificables. El resumen cuantitativo se presenta después de ejecutar las celdas de análisis.

## 2. Contexto del negocio

Un retailer multicanal vende en ocho regiones de Chile. La dirección desea comprender crecimiento, rentabilidad, clientes y riesgos promocionales.

## 3. Objetivos y preguntas

¿Cuánto vendemos y ganamos? ¿Qué productos, categorías, regiones y canales contribuyen más? ¿Dónde se deteriora el margen? ¿Qué acciones merecen una prueba comercial?

## 4. Herramientas

Python 3.11+, Pandas, NumPy, Matplotlib, Jupyter y pytest. El código reutilizable vive en `src/`.

## 5. Dataset

Transacciones sintéticas con semilla fija, 24 meses, pedidos multilínea y anomalías intencionales. `data/raw/` conserva la fuente generada; `data/processed/` contiene líneas validadas.

## 6. Diccionario de variables

Ver [diccionario completo](../reports/diccionario_datos.md). `descuento_pct` es una fracción entre 0 y 1. Los importes se expresan en CLP.

## 7. Importación de librerías

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.limpieza import limpiar
from src.kpis import enriquecer, resumen, tablas
from src.analisis import clp, pct
ROOT = Path.cwd()
assert (ROOT / 'src').is_dir(), 'Ejecute Jupyter desde la raíz del proyecto'

## 8. Carga de datos

In [2]:
raw = pd.read_csv(ROOT / 'data/raw/ventas_raw.csv', encoding='utf-8-sig')
raw.head()

,id_linea_venta,id_pedido,fecha_venta,id_cliente,id_producto,producto,categoria,region,canal_venta,cantidad,precio_unitario,costo_unitario,descuento_pct,estado_pedido
0,L000001,O000001,2024-06-14,C0671,P001,Audífonos,Tecnología,metropolitana,Marketplace,1,42600,39000,0.00,Completado
1,L000002,O000002,2025-02-20,C1063,P025,Secador,Belleza,metropolitana,Online,1,95800,51200,0.15,Completado
2,L000003,O000003,2025-11-10,C1123,P002,Teclado,Tecnología,metropolitana,Marketplace,1,186200,160200,0.15,Completado
3,L000004,O000004,2025-06-20,C1148,P002,Teclado,Tecnología,metropolitana,Online,1,182200,156300,0.00,Completado
4,L000005,O000005,2024-06-28,C0479,P002,Teclado,Tecnología,metropolitana,Tienda Física,1,183600,156400,0.00,Completado


## 9. Inspección inicial

In [3]:
print('Dimensiones:', raw.shape)
print('Columnas:', raw.columns.tolist())
raw.info()
display(raw.describe(include='all').T.head(14))

Dimensiones: (16193, 14)
Columnas: ['id_linea_venta', 'id_pedido', 'fecha_venta', 'id_cliente', 'id_producto', 'producto', 'categoria', 'region', 'canal_venta', 'cantidad', 'precio_unitario', 'costo_unitario', 'descuento_pct', 'estado_pedido']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16193 entries, 0 to 16192
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_linea_venta   16193 non-null  object 
 1   id_pedido        16193 non-null  object 
 2   fecha_venta      16193 non-null  object 
 3   id_cliente       16193 non-null  object 
 4   id_producto      16193 non-null  object 
 5   producto         16187 non-null  object 
 6   categoria        16187 non-null  object 
 7   region           16188 non-null  object 
 8   canal_venta      16193 non-null  object 
 9   cantidad         16193 non-null  int64  
 10  precio_unitario  16193 non-null  int64  
 11  costo_unitario   16193 non-null  int64  
 12

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_linea_venta,16193,16178,L000106,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_pedido,16193,10500,O000062,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fecha_venta,16193,673,2024-12-14,60,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_cliente,16193,1165,C0688,218,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_producto,16193,36,P001,1329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
producto,16187,36,Audífonos,1328,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,16187,6,Tecnología,5130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,16188,9,Metropolitana,6457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal_venta,16193,4,Online,7244,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cantidad,16193.0,NaN,NaN,NaN,1.416785,0.709553,0.0,1.0,1.0,2.0,4.0


## 10. Problemas de calidad detectados

Se inspeccionan nulos, duplicados y columnas con tipos incompatibles antes de filtrar.

In [4]:
display(raw.isna().sum().to_frame('nulos'))
print('Duplicados exactos:', raw.duplicated().sum())

,nulos
id_linea_venta,0
id_pedido,0
fecha_venta,0
id_cliente,0
id_producto,0
producto,6
categoria,6
region,5
canal_venta,0
cantidad,0


Duplicados exactos: 15


## 11. Limpieza y transformación

Normalización de espacios/capitalización, fechas y números. Atributos de catálogo faltantes se recuperan por ID de producto; fechas, precios y regiones ausentes no se imputan. La limpieza es idempotente.

In [5]:
clean, audit = limpiar(raw)
print(audit)
assert len(clean) == len(pd.read_csv(ROOT / 'data/processed/ventas_clean.csv'))
clean.head()

{'filas_iniciales': 16193, 'filas_finales': 16146, 'duplicados_eliminados': 15, 'nulos_antes': {'producto': 6, 'categoria': 6, 'region': 5}, 'campos_catalogo_recuperados': {'producto': 6, 'categoria': 6}, 'categorias_normalizadas': 36, 'registros_descartados': 32, 'motivos_descarte': {'campos_obligatorios_o_fecha': 10, 'cantidad_no_positiva': 5, 'precio_no_positivo': 5, 'costo_negativo': 5, 'descuento_fuera_0_1': 5, 'fecha_fuera_periodo': 5, 'cantidad_no_entera': 0, 'id_linea_repetido': 0, 'pedido_inconsistente_fecha_venta': 0, 'pedido_inconsistente_id_cliente': 0, 'pedido_inconsistente_region': 0, 'pedido_inconsistente_canal_venta': 2, 'pedido_inconsistente_estado_pedido': 0}}


,id_linea_venta,id_pedido,fecha_venta,id_cliente,id_producto,producto,categoria,region,canal_venta,cantidad,precio_unitario,costo_unitario,descuento_pct,estado_pedido
0,L000001,O000001,2024-06-14,C0671,P001,Audífonos,Tecnología,Metropolitana,Marketplace,1,42600,39000,0.00,Completado
1,L000002,O000002,2025-02-20,C1063,P025,Secador,Belleza,Metropolitana,Online,1,95800,51200,0.15,Completado
2,L000003,O000003,2025-11-10,C1123,P002,Teclado,Tecnología,Metropolitana,Marketplace,1,186200,160200,0.15,Completado
3,L000004,O000004,2025-06-20,C1148,P002,Teclado,Tecnología,Metropolitana,Online,1,182200,156300,0.00,Completado
4,L000005,O000005,2024-06-28,C0479,P002,Teclado,Tecnología,Metropolitana,Tienda Física,1,183600,156400,0.00,Completado


## 12. Validaciones

In [6]:
assert clean['id_linea_venta'].is_unique
assert clean['fecha_venta'].between('2024-01-01', '2025-12-31').all()
assert clean['cantidad'].gt(0).all()
assert clean['descuento_pct'].between(0, 1).all()
assert clean.notna().all().all()
print('Validaciones superadas:', len(clean), 'líneas')

Validaciones superadas: 16146 líneas


## 13. Feature engineering

Venta bruta = cantidad × precio; venta neta = bruta − descuento; utilidad bruta = venta neta − costo. Solo pedidos completados generan ingresos y costos reconocidos. El margen global usa sumas, no promedio de márgenes de línea.

In [7]:
ventas = enriquecer(clean)
ventas[['venta_bruta','descuento_monto','venta_neta','costo_total','utilidad_bruta','margen_bruto_pct']].head()

,venta_bruta,descuento_monto,venta_neta,costo_total,utilidad_bruta,margen_bruto_pct
0,42600.0,0.0,42600.0,39000.0,3600.0,8.450704
1,95800.0,14370.0,81430.0,51200.0,30230.0,37.123910
2,186200.0,27930.0,158270.0,160200.0,-1930.0,-1.219435
3,182200.0,0.0,182200.0,156300.0,25900.0,14.215148
4,183600.0,0.0,183600.0,156400.0,27200.0,14.814815


## 14. KPIs

In [8]:
kpi = resumen(ventas)
display(pd.Series(kpi, name='valor').to_frame())
t = tablas(ventas)

,valor
ventas_netas_clp,2.157751e+09
utilidad_bruta_clp,6.118160e+08
margen_bruto_pct,2.835434e+01
pedidos_completados,9.307000e+03
ticket_promedio_clp,2.318418e+05
unidades_vendidas,2.025600e+04
clientes_compradores,1.151000e+03
tasa_cancelacion_pct,7.032443e+00
tasa_devolucion_pct,4.160305e+00
pedidos_totales,1.048000e+04


## 15. Análisis temporal

**Pregunta:** ¿Cómo evolucionan las ventas y qué meses merecen investigación? **Decisión:** revisar meses de caída y comparar estacionalidad entre años.

In [9]:
display(t['mensual'][['mes','ventas_netas','utilidad_bruta','crecimiento_mom_pct']])
display(t['mensual'].pivot_table(index='numero_mes', columns='anio', values='ventas_netas'))

,mes,ventas_netas,utilidad_bruta,crecimiento_mom_pct
0,2024-01,65824980.0,16659080.0,NaN
1,2024-02,60498720.0,16885520.0,-8.091548
2,2024-03,78037245.0,21602045.0,28.989911
3,2024-04,84387690.0,22384990.0,8.137710
4,2024-05,75857080.0,21932680.0,-10.108832
5,2024-06,75223110.0,20326810.0,-0.835743
6,2024-07,77615660.0,20328360.0,3.180605
7,2024-08,81033045.0,20797945.0,4.402958
8,2024-09,81832065.0,21585965.0,0.986042
9,2024-10,89511785.0,24824585.0,9.384732


anio,2024,2025
numero_mes,,
1,65824980.0,78871790.0
2,60498720.0,74957810.0
3,78037245.0,87633165.0
4,84387690.0,84421765.0
5,75857080.0,100834105.0
6,75223110.0,90366560.0
7,77615660.0,96553050.0
8,81033045.0,89065670.0
9,81832065.0,95145340.0


![Ventas mensuales](../reports/figures/01_ventas_mensuales.png)

La variación mes a mes describe el cambio observado; no identifica sus causas.

## 16. Producto y categoría

**Pregunta:** ¿Qué se vende más y qué genera utilidad? **Decisión:** revisar productos de alta venta y margen bajo antes de ampliar promociones.

In [10]:
display(t['producto'].sort_values('ventas_netas', ascending=False).head(10))
display(t['producto'].sort_values('utilidad_bruta', ascending=False).head(10))
display(t['categoria'].sort_values('utilidad_bruta', ascending=False))

,id_producto,producto,categoria,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
1,P002,Teclado,Tecnología,281974260.0,17535260.0,1652,1277,6.218745
2,P003,Mouse,Tecnología,170364490.0,16087890.0,1540,1227,9.443218
19,P020,Escritorio,Oficina,84911245.0,19376145.0,515,387,22.819292
10,P011,Cafetera,Hogar,84115885.0,25066985.0,500,388,29.800536
20,P021,Cuaderno,Oficina,79722795.0,16830595.0,495,398,21.111396
17,P018,Balón,Deportes,75169780.0,19159780.0,536,414,25.488674
11,P012,Aspiradora,Hogar,72393880.0,34891580.0,470,376,48.196864
4,P005,Monitor,Tecnología,71895505.0,17986805.0,531,425,25.017983
0,P001,Audífonos,Tecnología,70568750.0,6216750.0,1673,1281,8.809494
23,P024,Archivador,Oficina,68485150.0,27029450.0,504,385,39.467607


,id_producto,producto,categoria,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
11,P012,Aspiradora,Hogar,72393880.0,34891580.0,470,376,48.196864
23,P024,Archivador,Oficina,68485150.0,27029450.0,504,385,39.467607
7,P008,Organizador,Hogar,68375530.0,26775530.0,440,344,39.159521
6,P007,Lámpara,Hogar,63152585.0,25149285.0,514,406,39.823049
10,P011,Cafetera,Hogar,84115885.0,25066985.0,500,388,29.800536
31,P032,Regadera,Jardín,52260815.0,23148715.0,482,383,44.294592
26,P027,Espejo,Belleza,61443340.0,23073140.0,443,345,37.551897
18,P019,Silla,Oficina,52760970.0,21916770.0,491,397,41.539740
3,P004,Webcam,Tecnología,48186270.0,21663670.0,475,381,44.958180
12,P013,Botella,Deportes,55002895.0,21286895.0,456,369,38.701408


,categoria,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
2,Hogar,372306435.0,140646635.0,2921,2177,37.777116
4,Oficina,380921365.0,111096065.0,2988,2217,29.165092
5,Tecnología,684200040.0,96029040.0,6374,4445,14.035229
3,Jardín,202345570.0,94257570.0,2189,1674,46.582473
0,Belleza,298121470.0,93006070.0,2806,2126,31.197374
1,Deportes,219856330.0,76780630.0,2978,2233,34.923093


![Ventas y utilidad por categoría](../reports/figures/10_ventas_utilidad_categoria.png)

## 17. Análisis regional

**Pregunta:** ¿Qué regiones concentran ventas y utilidad? **Decisión:** proteger la cobertura en zonas rentables y estudiar expansión.

In [11]:
display(t['region'].sort_values('ventas_netas', ascending=False))

,region,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
6,Metropolitana,859014535.0,243706135.0,8114,4181,28.370432
7,Valparaíso,338046400.0,92469400.0,3185,1643,27.354056
1,Biobío,283404115.0,82179615.0,2605,1372,28.997326
5,Maule,183606245.0,51899545.0,1731,900,28.266765
2,Coquimbo,143186575.0,39542475.0,1347,730,27.616049
4,Los Lagos,138592705.0,40346405.0,1277,615,29.111493
3,La Araucanía,124720590.0,36220590.0,1177,613,29.041388
0,Antofagasta,87180045.0,25451845.0,820,426,29.194577


![Ventas por región](../reports/figures/05_ventas_region.png)

## 18. Clientes

**Pregunta:** ¿Quiénes compran más y con qué frecuencia? `pedidos` cuenta identificadores únicos, no líneas.

In [12]:
display(t['clientes'].sort_values('ventas_netas', ascending=False).head(10))
print('Clientes recurrentes:', int(t['clientes']['pedidos'].gt(1).sum()))
print('Pedidos promedio por comprador:', round(t['clientes']['pedidos'].mean(), 2))

,id_cliente,ventas_netas,pedidos,ticket_promedio
656,C0688,32800855.0,137,239422.299270
474,C0494,25946860.0,115,225624.869565
309,C0324,21221200.0,100,212212.000000
528,C0552,20479225.0,90,227546.944444
73,C0082,20062770.0,75,267503.600000
974,C1021,18649515.0,90,207216.833333
838,C0880,17874825.0,71,251758.098592
717,C0754,16700615.0,68,245597.279412
67,C0076,16170065.0,73,221507.739726
607,C0637,14253970.0,57,250069.649123


Clientes recurrentes: 1042
Pedidos promedio por comprador: 8.09


## 19. Descuentos

**Pregunta:** ¿Cómo cambia el margen según el descuento? La comparación es descriptiva: mezcla de productos y clientes puede explicar parte de la diferencia.

In [13]:
display(t['descuentos'])

,rango_descuento,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
0,0%,874934200.0,288888200.0,7712,4592,33.018277
1,>0–10%,944097240.0,263322140.0,8925,5210,27.891421
2,>10–20%,306820210.0,58084110.0,3227,2157,18.930992
3,>20%,31899560.0,1521560.0,392,287,4.769846


![Descuentos y margen](../reports/figures/09_descuentos_margen.png)

## 20. Canales

**Pregunta:** ¿Qué canales aportan ventas y utilidad? **Decisión:** contrastar esta rentabilidad bruta con costos comerciales no modelados.

In [14]:
display(t['canal'].sort_values('ventas_netas', ascending=False))

,canal_venta,ventas_netas,utilidad_bruta,unidades,pedidos,margen_pct
1,Online,967857650.0,274849250.0,9081,4736,28.397694
2,Tienda Física,744408735.0,211862235.0,7057,3630,28.460471
0,Marketplace,445484825.0,125104525.0,4118,2114,28.082780


![Ventas por canal](../reports/figures/07_ventas_canal.png)

## 21. Principales hallazgos

Los seis hallazgos cuantificados se generan desde las tablas calculadas en [resumen ejecutivo](../reports/resumen_ejecutivo.md).

In [15]:
import json
hallazgos = json.loads((ROOT / 'reports/hallazgos.json').read_text(encoding='utf-8'))
for h in hallazgos: print(h['hallazgo'], '\nEvidencia:', h['evidencia'], '\n')

La categoría Hogar aporta $140.646.635 de utilidad bruta. 
Evidencia: Margen agregado: 37,8%; ventas: $372.306.435. 

Teclado combina ventas altas con margen de 6,2%. 
Evidencia: Ventas: $281.974.260; utilidad: $17.535.260. 

Metropolitana lidera las ventas regionales. 
Evidencia: Ventas: $859.014.535; utilidad: $243.706.135. 

Online es el mayor canal por ventas. 
Evidencia: Ventas: $967.857.650; margen: 28,4%. 

El mayor descenso mensual ocurrió en 2025-01. 
Evidencia: Variación frente al mes anterior: -32,2%; ventas: $78.871.790. 

Los descuentos altos se asocian con un margen distinto al de ventas sin descuento. 
Evidencia: Margen sin descuento: 33,0%; con descuento >20%: 4,8%. 



## 22. Recomendaciones de negocio

Priorizar categorías con mayor utilidad, revisar costos de productos populares con margen bajo, probar promociones con grupo de control y vigilar ventas/margen por región y canal.

In [16]:
for h in hallazgos: print('Acción:', h['accion'], '| KPI:', h['seguimiento'])

Acción: Priorizar surtido y disponibilidad de esta categoría. | KPI: Utilidad bruta y margen por categoría
Acción: Revisar costo de compra y límites de descuento de este producto. | KPI: Margen y utilidad por producto
Acción: Proteger cobertura logística y probar campañas incrementales en otras regiones. | KPI: Ventas y utilidad por región
Acción: Comparar costo comercial y adquisición antes de ampliar inversión. | KPI: Margen y ventas por canal
Acción: Investigar mezcla de productos, demanda y calendario promocional. | KPI: Crecimiento mensual de ventas
Acción: Probar promociones con grupo de control; esta comparación no demuestra causalidad. | KPI: Margen por rango de descuento


## 23. Limitaciones

Los datos son sintéticos. No se modelan impuestos, operación, marketing, logística ni intereses: la métrica es **utilidad bruta**, no utilidad neta empresarial. No inferir causalidad de correlaciones descriptivas.

## 24. Conclusiones finales

El caso permite identificar concentración comercial, meses débiles y presión de descuentos sobre el margen; las decisiones propuestas deben validarse con datos reales y pruebas controladas.

In [17]:
print(f"Ventas netas: {clp(kpi['ventas_netas_clp'])} CLP | Utilidad bruta: {clp(kpi['utilidad_bruta_clp'])} CLP | Margen: {pct(kpi['margen_bruto_pct'])}")

Ventas netas: $2.157.751.210 CLP | Utilidad bruta: $611.816.010 CLP | Margen: 28,4%
